###  G1

In [0]:
grant use catalog on catalog fk_1361 to fk_analyst

In [0]:
grant use schema, select on schema fk_1361.gold to fk_analyst

In [0]:
grant use catalog on catalog fk_1361 to fk_engineers;

grant use schema, select, modify on schema fk_1361.bronze to fk_engineers;
grant use schema, select, modify on schema fk_1361.silver to fk_engineers;
grant use schema, select, modify on schema fk_1361.gold to fk_engineers;
grant use schema, select, modify on schema fk_1361.ops to fk_engineers;

grant read volume, write volume on volume fk_1361.landing.raw to fk_engineers;

In [0]:
-- G2
CREATE TABLE IF NOT EXISTS fk_1361.ops.user_entitlements (
  user_email   STRING,
  allowed_city STRING,
  can_view_pii BOOLEAN
);

In [0]:
-- G3
CREATE OR REPLACE FUNCTION fk_1361.ops.mask_email(email STRING)
RETURNS STRING
RETURN CASE
  WHEN is_member('fk_admins') THEN email
  WHEN EXISTS (
    SELECT 1 FROM fk_1361.ops.user_entitlements
    WHERE user_email = current_user()
      AND can_view_pii = TRUE
  ) THEN email
  ELSE CONCAT(
    SUBSTRING(SPLIT_PART(email, '@', 1), 1, 1),
    '***@',
    SPLIT_PART(email, '@', 2)
  )
END;

CREATE OR REPLACE FUNCTION fk_1361.ops.mask_phone(phone STRING)
RETURNS STRING
RETURN CASE
  WHEN is_member('fk_admins') THEN phone
  WHEN EXISTS (
    SELECT 1 FROM fk_1361.ops.user_entitlements
    WHERE user_email = current_user()
      AND can_view_pii = TRUE
  ) THEN phone
  ELSE CONCAT('XXXXXX', RIGHT(phone, 4))
END;

ALTER TABLE fk_1361.gold.dim_customer
  ALTER COLUMN email SET MASK fk_1361.ops.mask_email;

ALTER TABLE fk_1361.gold.dim_customer
  ALTER COLUMN phone SET MASK fk_1361.ops.mask_phone;

In [0]:
-- G4
CREATE OR REPLACE FUNCTION fk_1361.ops.filter_city(city STRING)
RETURNS BOOLEAN
RETURN CASE
  WHEN is_member('fk_admins') THEN TRUE
  WHEN EXISTS (
    SELECT 1 FROM fk_1361.ops.user_entitlements
    WHERE user_email = current_user()
      AND (allowed_city = 'ALL' OR allowed_city = city)
  ) THEN TRUE
  ELSE FALSE
END;

ALTER TABLE fk_1361.gold.agg_daily_store_sales
  SET ROW FILTER fk_1361.ops.filter_city ON (city);